# Mass–Spring–Damper Cube + PPO Reinforcement Learning

이 notebook은 외부 물리 엔진 없이 8개의 질점과 스프링으로 cube를 만들고,
스프링 rest length를 행동으로 조절하여 +x 방향 locomotion을 학습하는 실습이다.

## 학습 순서
1. Cube topology 생성
2. Spring-damper force 계산
3. 중력 + ground contact
4. Gymnasium environment
5. Actor-Critic network
6. PPO training
7. Rollout visualization


In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.distributions import Normal
import gymnasium as gym
from gymnasium import spaces

SEED = 0
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Cube topology

8개 꼭짓점:
- (0,0,0), (1,0,0), ..., (1,1,1)

연결:
- edge
- face diagonal
- body diagonal

각 연결의 초기 길이를 rest length로 사용한다.


In [ ]:
def make_cube(side=0.4, z0=0.45):
    verts = np.array([
        [0,0,0], [1,0,0], [0,1,0], [1,1,0],
        [0,0,1], [1,0,1], [0,1,1], [1,1,1]
    ], dtype=np.float32)
    verts *= side
    verts[:, 0] -= side/2
    verts[:, 1] -= side/2
    verts[:, 2] += z0

    springs = []
    for i in range(8):
        for j in range(i+1, 8):
            diff = np.abs((verts[j] - verts[i]) / side)
            n_changed = np.sum(diff > 0.5)
            # cube의 두 vertex이면 거리 유형은 edge/sqrt(2)/sqrt(3)
            if n_changed in (1,2,3):
                springs.append((i,j))

    # 모든 vertex pair = 28개. cube에서는 edge + face diagonal + body diagonal.
    rest = np.array([np.linalg.norm(verts[j]-verts[i]) for i,j in springs], dtype=np.float32)
    return verts, springs, rest

x0, springs, rest = make_cube()
len(springs), x0


## 2. Physics simulator

각 spring에 대해

F = [k(L-L0) + c(v_rel · n)] n

을 계산한다.

바닥은 z=0 plane으로 두고 penalty force와 tangential damping을 적용한다.


In [ ]:
class MassSpringCube:
    def __init__(
        self,
        dt=0.005,
        substeps=4,
        mass=0.15,
        k=180.0,
        c=2.0,
        ground_k=1500.0,
        ground_c=15.0,
        friction=2.0,
        act_amp=0.20,
    ):
        self.dt = dt
        self.substeps = substeps
        self.mass = mass
        self.k = k
        self.c = c
        self.ground_k = ground_k
        self.ground_c = ground_c
        self.friction = friction
        self.act_amp = act_amp

        self.base_x, self.springs, self.base_rest = make_cube()
        self.n_springs = len(self.springs)

        # actuator는 edge spring만 사용
        self.actuator_ids = []
        for s, (i,j) in enumerate(self.springs):
            d = np.linalg.norm(self.base_x[j] - self.base_x[i])
            if np.isclose(d, 0.4, atol=1e-4):
                self.actuator_ids.append(s)

        self.n_act = len(self.actuator_ids)
        self.reset()

    def reset(self, noise=0.005):
        self.x = self.base_x.copy()
        self.x += np.random.randn(*self.x.shape).astype(np.float32) * noise
        self.v = np.zeros_like(self.x)
        return self.x.copy()

    def center_of_mass(self):
        return self.x.mean(axis=0)

    def _forces(self, action):
        F = np.zeros_like(self.x)
        F[:,2] -= self.mass * 9.81

        target_rest = self.base_rest.copy()
        for ai, sid in enumerate(self.actuator_ids):
            target_rest[sid] *= (1.0 + self.act_amp * float(action[ai]))

        for sid, (i,j) in enumerate(self.springs):
            d = self.x[j] - self.x[i]
            L = np.linalg.norm(d) + 1e-8
            n = d / L
            rel = np.dot(self.v[j] - self.v[i], n)
            f = (self.k * (L - target_rest[sid]) + self.c * rel) * n
            F[i] += f
            F[j] -= f

        # Ground: z < 0 이면 위쪽 penalty force
        penetration = np.minimum(self.x[:,2], 0.0)
        contact = penetration < 0
        F[contact,2] += -self.ground_k * penetration[contact]
        F[contact,2] += -self.ground_c * np.minimum(self.v[contact,2], 0.0)

        # 간단한 horizontal contact damping
        F[contact,0:2] += -self.friction * self.v[contact,0:2]
        return F

    def step(self, action):
        action = np.clip(np.asarray(action, dtype=np.float32), -1, 1)
        for _ in range(self.substeps):
            F = self._forces(action)
            a = F / self.mass
            # semi-implicit Euler
            self.v += a * self.dt
            self.x += self.v * self.dt

        return self.x.copy(), self.v.copy()


In [ ]:
sim = MassSpringCube()
print("springs:", sim.n_springs, "actuators:", sim.n_act)

xs = []
for t in range(300):
    a = np.zeros(sim.n_act, dtype=np.float32)
    x, v = sim.step(a)
    xs.append(x.copy())

print("final COM:", sim.center_of_mass())


## 3. 간단한 시각화


In [ ]:
def plot_cube(x, springs, ax=None, title="cube"):
    if ax is None:
        fig = plt.figure(figsize=(6,5))
        ax = fig.add_subplot(111, projection="3d")
    for i,j in springs:
        p, q = x[i], x[j]
        ax.plot([p[0],q[0]], [p[1],q[1]], [p[2],q[2]], alpha=0.35)
    ax.scatter(x[:,0], x[:,1], x[:,2], s=35)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_zlim(-0.05, 0.9)
    return ax

plot_cube(sim.x, sim.springs, title="Passive cube after settling")
plt.show()


## 4. Gymnasium environment

Observation은 translation invariance를 위해 COM 기준 상대 위치를 사용한다.
현재 Gymnasium API에 맞게 `reset()`은 `(obs, info)`,
`step()`은 `(obs, reward, terminated, truncated, info)`를 반환한다.


In [ ]:
class CubeLocomotionEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, episode_steps=400):
        super().__init__()
        self.sim = MassSpringCube()
        self.episode_steps = episode_steps
        self.t = 0

        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(self.sim.n_act,), dtype=np.float32
        )

        # rel_pos 24 + velocity 24 + COM_z 1 + COM_vel 3 = 52
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(52,), dtype=np.float32
        )

    def _obs(self):
        com = self.sim.center_of_mass()
        rel = self.sim.x - com[None,:]
        com_v = self.sim.v.mean(axis=0)
        obs = np.concatenate([
            rel.reshape(-1),
            self.sim.v.reshape(-1),
            np.array([com[2]], dtype=np.float32),
            com_v.astype(np.float32),
        ]).astype(np.float32)
        return obs

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            np.random.seed(seed)
        self.sim.reset()
        self.t = 0
        return self._obs(), {}

    def step(self, action):
        x_before = self.sim.center_of_mass()[0]
        self.sim.step(action)
        x_after = self.sim.center_of_mass()[0]

        self.t += 1
        com = self.sim.center_of_mass()

        forward = x_after - x_before
        energy = np.mean(np.square(action))
        low_height = max(0.0, 0.22 - float(com[2]))

        reward = 10.0 * forward - 0.01 * energy - 0.5 * low_height

        numerical_failure = (
            (not np.isfinite(self.sim.x).all())
            or np.abs(self.sim.x).max() > 20.0
            or com[2] < -0.2
        )

        terminated = bool(numerical_failure)
        truncated = bool(self.t >= self.episode_steps)

        info = {
            "com_x": float(com[0]),
            "com_z": float(com[2]),
            "forward": float(forward),
            "energy": float(energy),
        }
        return self._obs(), float(reward), terminated, truncated, info


In [ ]:
from gymnasium.utils.env_checker import check_env
check_env(CubeLocomotionEnv(), skip_render_check=True)

env = CubeLocomotionEnv()
obs, info = env.reset(seed=0)
print(obs.shape, env.action_space.shape)

for _ in range(10):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print(reward, info)


## 5. Actor-Critic network

연속 행동을 위해 Gaussian policy를 사용한다.
mean은 신경망에서 출력하고 log standard deviation은 학습 파라미터로 둔다.
샘플링한 pre-tanh action을 tanh로 제한한다.

수업 구현 단순화를 위해 PPO log-prob에서는 tanh correction까지 포함한다.


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self.log_std = nn.Parameter(torch.full((act_dim,), -0.5))

    def dist(self, obs):
        mu = self.actor(obs)
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def value(self, obs):
        return self.critic(obs).squeeze(-1)

    def act(self, obs):
        d = self.dist(obs)
        z = d.rsample()
        a = torch.tanh(z)
        logp = d.log_prob(z).sum(-1) - torch.log(1 - a.pow(2) + 1e-6).sum(-1)
        return a, logp, self.value(obs), z

    def evaluate(self, obs, z):
        d = self.dist(obs)
        a = torch.tanh(z)
        logp = d.log_prob(z).sum(-1) - torch.log(1 - a.pow(2) + 1e-6).sum(-1)
        entropy = d.entropy().sum(-1)
        return logp, entropy, self.value(obs)


## 6. GAE + PPO

실습 시간을 줄이기 위해 단일 환경 rollout을 사용한다.
GPU가 있어도 physics는 NumPy CPU에서 실행된다.

빠른 데모는 updates=20~50,
실제 학습은 updates=150~400 정도부터 실험한다.


In [ ]:
def compute_gae(rewards, dones, values, last_value, gamma=0.99, lam=0.95):
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    lastgaelam = 0.0
    for t in reversed(range(T)):
        next_value = last_value if t == T-1 else values[t+1]
        next_nonterminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * next_nonterminal - values[t]
        lastgaelam = delta + gamma * lam * next_nonterminal * lastgaelam
        adv[t] = lastgaelam
    returns = adv + values
    return adv, returns


In [ ]:
def train_ppo(
    updates=60,
    rollout_steps=1024,
    epochs=6,
    minibatch=256,
    lr=3e-4,
    clip_eps=0.2,
    gamma=0.99,
    lam=0.95,
    vf_coef=0.5,
    ent_coef=0.002,
):
    env = CubeLocomotionEnv()
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    net = ActorCritic(obs_dim, act_dim).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    obs, _ = env.reset(seed=SEED)
    history = {"return": [], "com_x": []}
    ep_return = 0.0

    for update in range(updates):
        O, Z, LP, R, D, V = [], [], [], [], [], []

        for _ in range(rollout_steps):
            ot = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                a, logp, val, z = net.act(ot)

            action = a.squeeze(0).cpu().numpy()
            next_obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            O.append(obs.copy())
            Z.append(z.squeeze(0).cpu().numpy())
            LP.append(float(logp.item()))
            R.append(float(reward))
            D.append(float(done))
            V.append(float(val.item()))

            ep_return += reward
            obs = next_obs

            if done:
                history["return"].append(ep_return)
                history["com_x"].append(info["com_x"])
                ep_return = 0.0
                obs, _ = env.reset()

        with torch.no_grad():
            ot = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            last_value = float(net.value(ot).item())

        adv, ret = compute_gae(
            np.array(R, dtype=np.float32),
            np.array(D, dtype=np.float32),
            np.array(V, dtype=np.float32),
            last_value,
            gamma, lam
        )
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        O = torch.tensor(np.array(O), dtype=torch.float32, device=device)
        Z = torch.tensor(np.array(Z), dtype=torch.float32, device=device)
        oldLP = torch.tensor(np.array(LP), dtype=torch.float32, device=device)
        ADV = torch.tensor(adv, dtype=torch.float32, device=device)
        RET = torch.tensor(ret, dtype=torch.float32, device=device)

        n = rollout_steps
        for _ in range(epochs):
            idx = torch.randperm(n, device=device)
            for start in range(0, n, minibatch):
                b = idx[start:start+minibatch]
                newLP, ent, value = net.evaluate(O[b], Z[b])
                ratio = (newLP - oldLP[b]).exp()

                s1 = ratio * ADV[b]
                s2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * ADV[b]
                policy_loss = -torch.min(s1, s2).mean()

                value_loss = (value - RET[b]).pow(2).mean()
                entropy = ent.mean()

                loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                opt.step()

        if (update+1) % 10 == 0:
            recent = history["return"][-10:]
            mean_r = np.mean(recent) if recent else float("nan")
            print(f"update {update+1:4d} | recent return {mean_r:8.3f}")

    return net, history


### 학습 실행

CPU에서도 실행할 수 있지만 수업 시간에는 `updates=20`으로 파이프라인만 확인하고,
과제로 더 오래 학습하는 방식을 권장한다.


In [ ]:
# 빠른 smoke test:
# policy, hist = train_ppo(updates=20, rollout_steps=512)

# 더 긴 실험:
# policy, hist = train_ppo(updates=200, rollout_steps=1024)


In [ ]:
# 학습 후 curve
# plt.figure(figsize=(7,4))
# plt.plot(hist["return"])
# plt.xlabel("Episode")
# plt.ylabel("Return")
# plt.title("PPO learning curve")
# plt.grid(alpha=0.3)
# plt.show()


## 7. Deterministic rollout

학습이 끝나면 policy mean을 tanh에 넣어 deterministic action으로 평가한다.


In [ ]:
def rollout(policy, steps=400):
    env = CubeLocomotionEnv(episode_steps=steps)
    obs, _ = env.reset(seed=123)
    frames = []
    rewards = []

    for _ in range(steps):
        frames.append(env.sim.x.copy())
        ot = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action = torch.tanh(policy.actor(ot)).squeeze(0).cpu().numpy()
        obs, r, term, trunc, info = env.step(action)
        rewards.append(r)
        if term or trunc:
            break

    return np.array(frames), rewards, info

# frames, rewards, info = rollout(policy)
# print("final COM x =", info["com_x"])


In [ ]:
# 시작/끝 자세 비교
# fig = plt.figure(figsize=(11,4))
# ax1 = fig.add_subplot(121, projection="3d")
# ax2 = fig.add_subplot(122, projection="3d")
# plot_cube(frames[0], env.sim.springs if 'env' in globals() else make_cube()[1], ax1, "start")
# plot_cube(frames[-1], make_cube()[1], ax2, "end")
# plt.show()


## 8. 실험 아이디어

1. damping `c = 0.2 / 1.0 / 3.0`
2. spring stiffness `k = 80 / 180 / 400`
3. action amplitude `0.1 / 0.2 / 0.35`
4. energy penalty 제거
5. diagonal spring 제거
6. observation에서 velocity 제거
7. mass / friction domain randomization

### 생각해볼 질문
- 큰 k에서 왜 dt를 줄여야 하는가?
- velocity를 observation에서 빼면 policy가 상태를 충분히 알 수 있는가?
- reward에 forward displacement만 두면 어떤 비정상 동작이 생길 수 있는가?
